In [1]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class ReportMetadata:
    id: str
    title: str
    subtitle: str | None = None
    author: str | None = None
    organization: str | None = None
    date: str | None = None
    version: str = "1.0"
    status: str = "draft"
    confidentiality: str | None = None

@dataclass
class PresentationConfig:
    aspect_ratio: str = "16:9"
    theme: str = "technical"

    title_font: str = "Aptos Display"
    body_font: str = "Aptos"
    monospace_font: str = "Aptos Mono"

    show_slide_numbers: bool = True
    show_footer: bool = True
    footer_text: str | None = None

@dataclass
class Metric:
    id: str
    label: str
    value: Any
    display_value: str
    units: str | None = None
    interpretation: str | None = None

@dataclass
class Slide:
    id: str
    type: str
    title: str
    notes: str | None = None

@dataclass
class Figure:
    path: str
    caption: str | None = None
    alt_text: str | None = None
    width: str | None = None

@dataclass
class FigureSlide(Slide):
    figure: Figure | None = None
    annotations: list[dict] = field(default_factory=list)

@dataclass
class TableColumn:
    key: str
    label: str

@dataclass
class Table:
    columns: list[TableColumn]
    rows: list[dict]

@dataclass
class TableSlide(Slide):
    table: Table | None = None

@dataclass
class TwoColumn:
    heading: str | None = None
    paragraphs: list[str] = field(default_factory=list)
    bullets: list[str] = field(default_factory=list)

@dataclass
class TwoColumnSlide(Slide):
    left: TwoColumn = field(default_factory=TwoColumn)
    right: TwoColumn = field(default_factory=TwoColumn)

@dataclass
class MetricsSlide(Slide):
    metrics: list[Metric] = field(default_factory=list)

@dataclass
class Finding:
    severity: str
    title: str
    text: str

@dataclass
class FindingsSlide(Slide):
    findings: list[Finding] = field(default_factory=list)

@dataclass
class Section:
    id: str
    type: str
    title: str
    purpose: str | None = None
    slides: list[Slide] = field(default_factory=list)

@dataclass
class ExecutiveSummary:
    overview: str
    key_metrics: list[Metric] = field(default_factory=list)
    headline_findings: list[str] = field(default_factory=list)


@dataclass
class Report:
    metadata: ReportMetadata
    presentation: PresentationConfig
    executive_summary: ExecutiveSummary
    sections: list[Section] = field(default_factory=list)
    appendix: list[Slide] = field(default_factory=list)
    references: list[dict] = field(default_factory=list)
    assets: dict = field(default_factory=dict)

In [2]:
class ReportParser:

    def parse(self, data: dict) -> Report:

        return Report(
            metadata=self._parse_metadata(data["report"]),

            presentation=self._parse_presentation(
                data["presentation"]
            ),

            executive_summary=self._parse_summary(
                data["executive_summary"]
            ),

            sections=[
                self._parse_section(section)
                for section in data["sections"]
            ],

            appendix=[
                self._parse_slide(item)
                for item in data.get("appendix", [])
            ],

            references=data.get("references", []),

            assets=data.get("assets", {})
        )

    # -------------------------
    # Metadata
    # -------------------------

    def _parse_metadata(self, data: dict) -> ReportMetadata:

        return ReportMetadata(
            id=data["id"],
            title=data["title"],
            subtitle=data.get("subtitle"),
            author=data.get("author"),
            organization=data.get("organization"),
            date=data.get("date"),
            version=data.get("version", "1.0"),
            status=data.get("status", "draft"),
            confidentiality=data.get("confidentiality")
        )

    # -------------------------
    # Presentation
    # -------------------------

    def _parse_presentation(self, data: dict) -> PresentationConfig:

        fonts = data.get("fonts", {})

        return PresentationConfig(
            aspect_ratio=data.get("aspect_ratio", "16:9"),
            theme=data.get("theme", "technical"),

            title_font=fonts.get(
                "title",
                "Aptos Display"
            ),

            body_font=fonts.get(
                "body",
                "Aptos"
            ),

            monospace_font=fonts.get(
                "monospace",
                "Aptos Mono"
            ),

            show_slide_numbers=data.get(
                "show_slide_numbers",
                True
            ),

            show_footer=data.get(
                "show_footer",
                True
            ),

            footer_text=data.get("footer_text")
        )

    # -------------------------
    # Executive summary
    # -------------------------

    def _parse_summary(self, data: dict) -> ExecutiveSummary:

        metrics = [
            Metric(
                id=m["id"],
                label=m["label"],
                value=m["value"],
                display_value=m["display_value"],
                units=m.get("units"),
                interpretation=m.get("interpretation")
            )
            for m in data.get("key_metrics", [])
        ]

        return ExecutiveSummary(
            overview=data["overview"],
            key_metrics=metrics,
            headline_findings=data.get(
                "headline_findings",
                []
            )
        )

    # -------------------------
    # Sections
    # -------------------------

    def _parse_section(self, data: dict) -> Section:

        return Section(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            purpose=data.get("purpose"),
            slides=[
                self._parse_slide(slide)
                for slide in data.get("slides", [])
            ]
        )

    # -------------------------
    # Slide dispatcher
    # -------------------------

    def _parse_slide(self, data: dict) -> Slide:

        slide_type = data["type"]

        if slide_type == "figure":
            return self._parse_figure_slide(data)

        if slide_type == "table":
            return self._parse_table_slide(data)

        if slide_type == "metrics":
            return self._parse_metrics_slide(data)

        if slide_type == "two_column":
            return self._parse_two_column_slide(data)

        if slide_type == "findings":
            return self._parse_findings_slide(data)

        # Temporary generic handling
        return Slide(
            id=data["id"],
            type=slide_type,
            title=data["title"],
            notes=data.get("notes")
        )

    # -------------------------
    # Figure
    # -------------------------

    def _parse_figure_slide(self, data: dict) -> FigureSlide:

        figure_data = data["figure"]

        figure = Figure(
            path=figure_data["path"],
            caption=figure_data.get("caption"),
            alt_text=figure_data.get("alt_text"),
            width=figure_data.get("width")
        )

        return FigureSlide(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            notes=data.get("notes"),
            figure=figure,
            annotations=data.get("annotations", [])
        )

    # -------------------------
    # Table
    # -------------------------

    def _parse_table_slide(self, data: dict) -> TableSlide:

        table_data = data["table"]

        columns = [
            TableColumn(
                key=column["key"],
                label=column["label"]
            )
            for column in table_data["columns"]
        ]

        table = Table(
            columns=columns,
            rows=table_data["rows"]
        )

        return TableSlide(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            notes=data.get("notes"),
            table=table
        )

    # -------------------------
    # Metrics
    # -------------------------

    def _parse_metrics_slide(self, data: dict) -> MetricsSlide:

        metrics = [
            Metric(
                id=str(i),
                label=m["label"],
                value=m.get("value"),
                display_value=m["value"],
                interpretation=m.get("detail")
            )
            for i, m in enumerate(data.get("metrics", []))
        ]

        return MetricsSlide(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            notes=data.get("notes"),
            metrics=metrics
        )

    # -------------------------
    # Two-column
    # -------------------------

    def _parse_two_column_slide(self, data: dict) -> TwoColumnSlide:

        def parse_column(column):

            return TwoColumn(
                heading=column.get("heading"),
                paragraphs=column.get("paragraphs", []),
                bullets=column.get("bullets", [])
            )

        return TwoColumnSlide(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            notes=data.get("notes"),
            left=parse_column(data.get("left", {})),
            right=parse_column(data.get("right", {}))
        )

    # -------------------------
    # Findings
    # -------------------------

    def _parse_findings_slide(self, data: dict) -> FindingsSlide:

        findings = [
            Finding(
                severity=f["severity"],
                title=f["title"],
                text=f["text"]
            )
            for f in data.get("findings", [])
        ]

        return FindingsSlide(
            id=data["id"],
            type=data["type"],
            title=data["title"],
            notes=data.get("notes"),
            findings=findings
        )
    

In [3]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_SHAPE

class PptxRenderer:

    def __init__(self, report: Report):
        self.report = report
        self.prs = Presentation()
        # Standard Size
        self.prs.slide_width = Inches(13.333)
        self.prs.slide_height = Inches(7.5)

    def render(self):
        for section in self.report.sections:
            for slide in section.slides:
                self._render_slide(slide)
        return self.prs

    def _new_slide(self):
        return self.prs.slides.add_slide(self.prs.slide_layouts[6])

    def _add_title(self, slide, title):
        textbox = slide.shapes.add_textbox(Inches(0.5),Inches(0.25),Inches(12.3),Inches(0.6))
        text_frame = textbox.text_frame
        text_frame.text = title
        paragraph = text_frame.paragraphs[0]
        paragraph.font.size = Pt(28)
        paragraph.font.bold = True

    def _add_metrics(self, slide, metrics):
        if not metrics:
            return
        margin = Inches(0.6)
        top = Inches(1.5)
        available_width = (self.prs.slide_width- 2 * margin)
        gap = Inches(0.25)
        card_width = (available_width- gap * (len(metrics) - 1)) / len(metrics)
        card_height = Inches(2.0)
        for i, metric in enumerate(metrics):
            left = (margin+ i * (card_width + gap))
            shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left,top,card_width,card_height)
            shape.text_frame.clear()
            # Label
            paragraph = shape.text_frame.paragraphs[0]
            paragraph.text = metric.label
            paragraph.font.size = Pt(14)
            paragraph.font.bold = True
            # Value
            paragraph = shape.text_frame.add_paragraph()
            paragraph.text = str(metric.display_value)
            paragraph.font.size = Pt(30)
            paragraph.font.bold = True

    def _add_footer(self, slide):
        if not self.report.presentation.show_footer:
            return
        footer = self.report.presentation.footer_text
        if not footer:
            return
        textbox = slide.shapes.add_textbox(Inches(0.6),Inches(7.05),Inches(12.1),Inches(0.25))
        paragraph = textbox.text_frame.paragraphs[0]
        paragraph.text = footer
        paragraph.font.size = Pt(9)

    def _add_figure(self, slide, figure):
        image_path = Path(figure.path)

        try: 
            # Default image position
            left = Inches(0.75)
            top = Inches(1.5)
            width = Inches(11.8)
            # Use width specified in JSON if provided
            if figure.width == 'full':
                width = Inches(11.8)
            elif figure.width is not None:
                width = Inches(figure.width)
    

            slide.shapes.add_picture(str(image_path),left,top,width=width)
            # Caption
            if figure.caption:
                textbox = slide.shapes.add_textbox(Inches(0.75),Inches(6.8),Inches(11.8),Inches(0.3))
                textbox.text_frame.text = figure.caption
        except Exception as e: 
            raise Warning(f'{e}: {image_path}')

    def _add_table(self, slide, table):
        columns = table.columns
        rows = table.rows
        n_rows = len(rows) + 1 
        n_cols = len(columns)

        # Table position
        left = Inches(0.5)
        top = Inches(1.5)
        width = Inches(12.3)
        height = Inches(5.2)
        ppt_table = slide.shapes.add_table(n_rows,n_cols,left,top,width,height).table

        # Header
        for col_idx, column in enumerate(columns):
            cell = ppt_table.cell(0,col_idx)
            cell.text = column.label
        # Data
        for row_idx, row in enumerate(rows, start=1):
            for col_idx, column in enumerate(columns):
                value = row.get(column.key, "")
                cell = ppt_table.cell(row_idx,col_idx)
                cell.text = str(value)

    def _render_slide(self, slide):
        renderers = {
            "figure": self._render_figure_slide,
            "table": self._render_table_slide,
            "metrics": self._render_metrics_slide,
            "two_column": self._render_two_column_slide,
            "findings": self._render_findings_slide,
        }
        renderer = renderers.get(slide.type)
        if renderer is None:
            return self._render_generic_slide(slide)
        renderer(slide)

    def _render_figure_slide(self, slide: FigureSlide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        self._add_figure(ppt_slide,slide.figure)
        self._add_footer(ppt_slide)

    def _render_table_slide(self, slide: TableSlide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        self._add_table(ppt_slide,slide.table)
        self._add_footer(ppt_slide)

    def _render_metrics_slide(self, slide: MetricsSlide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        self._add_metrics(ppt_slide,slide.metrics)
        self._add_footer(ppt_slide)

    def _render_two_column_slide(self, slide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        # self._add_two_columns(...)
        self._add_footer(ppt_slide)

    def _render_findings_slide(self, slide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        # self._add_findings(...)
        self._add_footer(ppt_slide)

    def _render_generic_slide(self, slide):
        ppt_slide = self._new_slide()
        self._add_title(ppt_slide,slide.title)
        self._add_footer(ppt_slide)

    def save(self, path):
        self.prs.save(path)

    



In [4]:
import json
from pathlib import Path

json_path = Path("report.json")

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

data.keys()


dict_keys(['schema_version', 'report', 'presentation', 'executive_summary', 'sections', 'appendix', 'references', 'assets'])

In [5]:
parser = ReportParser()
report = parser.parse(data)
report.metadata

ReportMetadata(id='network_reliability_analysis_001', title='Network Reliability Analysis', subtitle='Monte Carlo Simulation, Sensitivity Analysis, and Critical-Component Identification', author='Andrew Sheldon', organization='Operations Research', date='2026-08-13', version='1.0', status='draft', confidentiality='Internal Research')

In [6]:
for section in report.sections:

    print(section.title)

    for slide in section.slides:
        print(f"   {slide.type:15}    "f"{slide.title}")

1. Problem Definition
   two_column         Problem Overview
   figure             Network Structure
2. Methodology
   model              Network Reliability Model
   process            Monte Carlo Simulation Procedure
   table              Simulation Configuration
3. Results
   metrics            Primary Results
   figure             Estimated Reliability Across Simulation Batches
   figure             Monte Carlo Convergence
4. Sensitivity Analysis
   table              Component Influence Ranking
   figure             Reliability Sensitivity Surface
   findings           Sensitivity Findings
5. Discussion
   two_column         Interpretation of Results
   findings           Model Limitations
6. Recommendations
   recommendations    Recommended Next Steps
   summary            Decision Summary


In [7]:
renderer = PptxRenderer(report)
renderer.render()
renderer.save("report.pptx")